# Immunity versus infection — genes, shared lead variants, and their directions

Follow-on to `01_directionality_checks.ipynb`, narrowed to the axis the referee actually names:

> I would consider it more plausible that a lot of the lead variant alleles that increased, for
> example, risk of autoimmunity, would decrease risk of infection. Is it really the case there are
> not other examples of balancing selection?

Three steps, in order:

1. **Genes pleiotropic across the two therapeutic areas** — genes with at least one credible set in
   an `immune system disease` trait and at least one in an `infectious disease` trait.
2. **Same lead variant on both sides** — within those genes, the lead variants that carry both an
   immune and an infection credible set. Only there can the two directions be compared, because only
   there do they refer to the same effect allele.
3. **A few representatives**, reported association by association.

## Classes

Therapeutic areas from `disease.therapeuticAreas` in Open Targets 25.06:
`EFO_0000540` (immune system disease) and `EFO_0005741` (infectious disease). This is deliberately
the therapeutic-area axis, not the `autoimmune disease` ontology branch — the immune-system area is
broader than autoimmunity (it includes allergy, immunodeficiency and inflammatory disease) and,
unlike the `EFO_0005140` branch, it does contain type 1 diabetes, celiac disease and psoriasis.

Where a term carries both areas it is counted as **infection** (only one such term has data here:
`EFO_0007429`, peritonsillar abscess).

## Alleles, not variants

As in notebook 01: every direction names an **effect allele**, the alternative allele of
`chrom_pos_ref_alt`, which is what studies are harmonised to at ingestion. Credible sets sharing a
`variantId` share an effect allele, which is exactly why step 2 restricts the direction comparison to
shared lead variants — two different lead variants in one gene carry two different alleles and their
signs are not comparable without LD phase.

Signed effect: `rescaledStatistics.directionOfEffect × rescaledStatistics.absEstimatedBeta`.
Concordance: the paper's definition (Methods), largest proportion of same-direction effects per
variant over its credible sets reporting a beta.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 300)

INTERMEDIATE = "../../../data/intermediate_files/"
RELEASE = "../../../data/25.06/"

IMMUNE_AREA = "EFO_0000540"  # immune system disease
INFECTION_AREA = "EFO_0005741"  # infectious disease

In [2]:
cs = (
    ds.dataset(INTERMEDIATE + "qualifying_credible_sets", format="parquet")
    .to_table(
        columns=[
            "studyId",
            "studyLocusId",
            "variantId",
            "variant",
            "diseaseIds",
            "originalBeta",
            "originalStandardError",
            "rescaledStatistics",
            "studyStatistics",
            "nCases",
            "nControls",
        ]
    )
    .to_pandas()
)

rescaled = pd.DataFrame(list(cs["rescaledStatistics"]))
cs["beta"] = rescaled["directionOfEffect"].to_numpy() * rescaled["absEstimatedBeta"].to_numpy()
cs["se"] = rescaled["estimatedSE"].to_numpy()
cs["trait"] = pd.DataFrame(list(cs["studyStatistics"]))["trait"].to_numpy()
cs["effectAllele"] = cs["variant"].apply(lambda v: v["alt"])
cs["otherAllele"] = cs["variant"].apply(lambda v: v["ref"])
cs = cs.drop(columns=["variant", "rescaledStatistics", "studyStatistics"])

disease = pd.read_parquet(RELEASE + "output/disease", columns=["id", "name", "therapeuticAreas"])
DISEASE_NAME = dict(zip(disease["id"], disease["name"]))
AREAS_OF = {i: set(t) if t is not None else set() for i, t in zip(disease["id"], disease["therapeuticAreas"])}

genes = pd.read_parquet(INTERMEDIATE + "list_of_prioritised_genes_per_CS.parquet")[["studyLocusId", "geneId"]]
target = pd.read_parquet(RELEASE + "output/target", columns=["id", "approvedSymbol"])
SYMBOL = dict(zip(target["id"], target["approvedSymbol"]))

print(f"qualifying credible sets: {len(cs):,}")
print(
    f"of these, with an L2G-prioritised gene: "
    f"{genes.loc[genes['studyLocusId'].isin(set(cs['studyLocusId'])), 'studyLocusId'].nunique():,}"
)

qualifying credible sets: 70,618
of these, with an L2G-prioritised gene: 68,675


In [3]:
def paper_concordance(frame: pd.DataFrame) -> float:
    """Largest proportion of same-direction effects, over credible sets reporting a beta."""
    reported = frame.drop_duplicates("studyLocusId")
    reported = reported.loc[reported["originalBeta"].notna(), "beta"].dropna()
    if len(reported) == 0:
        return 1.0
    positive = float((reported > 0).mean())
    return max(positive, 1.0 - positive)


def median_beta(betas: pd.Series) -> float:
    """Median of the reported betas, NaN when none are reported."""
    reported = betas.dropna()
    return float(reported.median()) if len(reported) else float("nan")


def sign_of(betas: pd.Series) -> str:
    """Majority sign of a set of effect-allele betas: '+', '-' or 'mixed'."""
    reported = betas.dropna()
    positive, negative = int((reported > 0).sum()), int((reported < 0).sum())
    if positive > negative:
        return "+"
    if negative > positive:
        return "-"
    return "mixed"


classified = cs.explode("diseaseIds").rename(columns={"diseaseIds": "diseaseId"}).dropna(subset=["diseaseId"])
classified["diseaseName"] = classified["diseaseId"].map(DISEASE_NAME)
classified["class"] = np.where(
    classified["diseaseId"].map(lambda d: INFECTION_AREA in AREAS_OF.get(d, set())),
    "infection",
    np.where(classified["diseaseId"].map(lambda d: IMMUNE_AREA in AREAS_OF.get(d, set())), "immune", None),
)
classified = classified[classified["class"].notna()]

joint_loci = classified.groupby("studyLocusId")["class"].nunique()
joint_loci = set(joint_loci[joint_loci > 1].index)
classified["joint_study"] = classified["studyLocusId"].isin(joint_loci)

both_areas = sorted(
    {d for d in classified["diseaseId"].unique() if {IMMUNE_AREA, INFECTION_AREA} <= AREAS_OF.get(d, set())}
)
print(
    f"credible set x disease rows — immune: {(classified['class'] == 'immune').sum():,}, "
    f"infection: {(classified['class'] == 'infection').sum():,}"
)
print(
    f"distinct diseases — immune: {classified.loc[classified['class'] == 'immune', 'diseaseId'].nunique()}, "
    f"infection: {classified.loc[classified['class'] == 'infection', 'diseaseId'].nunique()}"
)
print(f"terms carrying both areas (counted as infection): {[f'{d} ({DISEASE_NAME.get(d)})' for d in both_areas]}")
print(
    f"\ncredible sets mapping to an immune AND an infection disease at once "
    f"(trans-disease / MTAG studies): {len(joint_loci)}"
)
print(classified.loc[classified["joint_study"], ["studyId", "trait"]].drop_duplicates().head(8).to_string(index=False))

credible set x disease rows — immune: 9,453, infection: 1,143
distinct diseases — immune: 107, infection: 57
terms carrying both areas (counted as infection): ['EFO_0007429 (peritonsillar abscess)']

credible sets mapping to an immune AND an infection disease at once (trans-disease / MTAG studies): 81
     studyId                                                                                                 trait
GCST90255367                                                        Severe COVID-19 or rheumatoid arthritis (MTAG)
GCST90255371                                       COVID-19 hospitalization or systemic lupus erythematosus (MTAG)
  GCST004099 B-cell malignancies (chronic lymphocytic leukemia, Hodgkin lymphoma or multiple myeloma) (pleiotropy)
GCST90255368                                               COVID-19 hospitalization or rheumatoid arthritis (MTAG)
GCST90243963                                    Severe COVID-19 or systemic lupus erythematosus (inverse variance)
GCST902

# Step 1 — genes pleiotropic across the two areas

One gene per credible set is the L2G-prioritised gene, the same assignment the gene pleiotropy score
uses. A gene qualifies when it has at least one immune credible set **and** at least one infection
credible set — the credible sets need not share a lead variant at this step.

In [4]:
with_genes = classified.merge(genes, on="studyLocusId", how="inner")
with_genes["symbol"] = with_genes["geneId"].map(SYMBOL)

gene_table = (
    with_genes.groupby(["geneId", "symbol"])
    .apply(
        lambda g: pd.Series(
            {
                "immune_credible_sets": g.loc[g["class"] == "immune", "studyLocusId"].nunique(),
                "infection_credible_sets": g.loc[g["class"] == "infection", "studyLocusId"].nunique(),
                "immune_diseases": g.loc[g["class"] == "immune", "diseaseId"].nunique(),
                "infection_diseases": g.loc[g["class"] == "infection", "diseaseId"].nunique(),
                "lead_variants": g["variantId"].nunique(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
gene_table = gene_table[(gene_table["immune_credible_sets"] > 0) & (gene_table["infection_credible_sets"] > 0)]
gene_table = gene_table.sort_values(["immune_credible_sets", "infection_credible_sets"], ascending=False)
gene_table.to_csv(INTERMEDIATE + "immunity_infection_genes-r1.csv", index=False)

genes_excluding_joint = with_genes[~with_genes["joint_study"]].groupby("geneId")["class"].nunique()
print(f"genes with at least one credible set in each area: {len(gene_table)}")
print(
    f"  of which still qualify after dropping trans-disease / MTAG credible sets: "
    f"{int((genes_excluding_joint > 1).sum())}"
)
print(f"of {with_genes['geneId'].nunique():,} genes with any immune or infection credible set")
print()
print(gene_table.head(25).to_string(index=False))

genes with at least one credible set in each area: 207
  of which still qualify after dropping trans-disease / MTAG credible sets: 167
of 2,222 genes with any immune or infection credible set

         geneId  symbol  immune_credible_sets  infection_credible_sets  immune_diseases  infection_diseases  lead_variants
ENSG00000113302   IL12B                   107                        2               15                   1             51
ENSG00000138378   STAT4                   102                        2               24                   2             36
ENSG00000105397    TYK2                    82                       23               15                   1             13
ENSG00000128604    IRF5                    81                        3               16                   1             24
ENSG00000118503 TNFAIP3                    75                        1               14                   1             48
ENSG00000137507  LRRC32                    65                        

# Step 2 — the same lead variant on both sides

A gene can reach both areas through two different lead variants, and then the two directions refer to
two different alleles and cannot be compared. Restricting to lead variants that themselves carry an
immune credible set **and** an infection credible set makes the comparison allele-exact.

One more exclusion is needed first. Trans-disease and MTAG studies — "COVID-19 infection or systemic
lupus erythematosus (MTAG)", "Psoriasis or type 2 diabetes (trans-disease meta-analysis)" — map a
single credible set to a disease in each area, so the *same* beta appears on both sides and the pair
is concordant by construction. Those credible sets are dropped from the direction comparison; the
count with and without them is reported, because it is large.

In [5]:
comparable = with_genes[~with_genes["joint_study"]]
print(
    f"credible set x disease rows: {len(with_genes):,} total, "
    f"{len(comparable):,} after dropping trans-disease / MTAG credible sets"
)

shared_including_joint = with_genes.groupby(["geneId", "variantId"]).apply(
    lambda g: (g["class"] == "immune").any() and (g["class"] == "infection").any(), include_groups=False
)
print(
    f"gene x lead-variant pairs carrying both areas: "
    f"{int(shared_including_joint.sum())} including trans-disease credible sets, "
    f"and the count below after dropping them"
)

shared = (
    comparable.groupby(["geneId", "symbol", "variantId", "effectAllele", "otherAllele"])
    .apply(
        lambda g: pd.Series(
            {
                "immune_credible_sets": g.loc[g["class"] == "immune", "studyLocusId"].nunique(),
                "infection_credible_sets": g.loc[g["class"] == "infection", "studyLocusId"].nunique(),
                "immune_diseases": g.loc[g["class"] == "immune", "diseaseId"].nunique(),
                "infection_diseases": g.loc[g["class"] == "infection", "diseaseId"].nunique(),
                "immune_sign": sign_of(g.loc[g["class"] == "immune", "beta"]),
                "infection_sign": sign_of(g.loc[g["class"] == "infection", "beta"]),
                "immune_median_beta": median_beta(g.loc[g["class"] == "immune", "beta"]),
                "infection_median_beta": median_beta(g.loc[g["class"] == "infection", "beta"]),
                "infection_disease_names": "; ".join(
                    sorted(set(g.loc[g["class"] == "infection", "diseaseName"].dropna()))
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
shared = shared[(shared["immune_credible_sets"] > 0) & (shared["infection_credible_sets"] > 0)].copy()

shared["verdict"] = np.where(
    (shared["immune_sign"] == "mixed") | (shared["infection_sign"] == "mixed"),
    "undetermined",
    np.where(shared["immune_sign"] == shared["infection_sign"], "concordant", "discordant"),
)
shared["variant_concordance_paper_formula"] = (
    shared["variantId"].map(cs.groupby("variantId").apply(paper_concordance, include_groups=False)).round(3)
)
shared = shared.sort_values(["immune_credible_sets", "infection_credible_sets"], ascending=False)
shared.to_csv(INTERMEDIATE + "immunity_infection_shared_variants-r1.csv", index=False)

print(f"gene x lead-variant pairs carrying both areas: {len(shared)}")
print(f"  distinct genes:         {shared['geneId'].nunique()}")
print(f"  distinct lead variants: {shared['variantId'].nunique()}")
determined = shared[shared["verdict"] != "undetermined"]
print(f"\nverdicts (both sides have at least one reported beta):")
print(shared["verdict"].value_counts().to_string())
print(f"discordant share of determined: {(shared['verdict'] == 'discordant').sum() / max(len(determined), 1):.1%}")

credible set x disease rows: 10,510 total, 10,349 after dropping trans-disease / MTAG credible sets


gene x lead-variant pairs carrying both areas: 122 including trans-disease credible sets, and the count below after dropping them


gene x lead-variant pairs carrying both areas: 59
  distinct genes:         44
  distinct lead variants: 58

verdicts (both sides have at least one reported beta):
verdict
concordant      34
discordant      16
undetermined     9
discordant share of determined: 32.0%


In [6]:
print("all gene x lead-variant pairs, most immune credible sets first:")
print(
    shared[
        [
            "symbol",
            "variantId",
            "effectAllele",
            "immune_credible_sets",
            "infection_credible_sets",
            "immune_sign",
            "infection_sign",
            "immune_median_beta",
            "infection_median_beta",
            "verdict",
            "variant_concordance_paper_formula",
            "infection_disease_names",
        ]
    ]
    .head(45)
    .to_string(index=False)
)

all gene x lead-variant pairs, most immune credible sets first:
         symbol        variantId effectAllele  immune_credible_sets  infection_credible_sets immune_sign infection_sign  immune_median_beta  infection_median_beta      verdict  variant_concordance_paper_formula                                                                                                                                   infection_disease_names
           TYK2  19_10352442_G_C            C                    38                        3           -          mixed           -0.283324                    NaN undetermined                              0.974                                                                                                                                                  COVID-19
         TNRC18    7_5397122_C_T            T                    34                        1           +              +            0.605567               0.320853   concordant                              0

### Which infections are actually on the other side

The infection half of the corpus is thin and skewed, and that shapes everything above.

In [7]:
infection_side = comparable[
    (comparable["class"] == "infection") & comparable["variantId"].isin(set(shared["variantId"]))
]
infection_profile = (
    infection_side.groupby(["diseaseId", "diseaseName"])
    .agg(credible_sets=("studyLocusId", "nunique"), lead_variants=("variantId", "nunique"))
    .reset_index()
    .sort_values("credible_sets", ascending=False)
)
print(infection_profile.to_string(index=False))
print(
    "\ncaveat: EFO_0000183 (Hodgkins lymphoma) and the other lymphoma terms carry the infectious "
    "disease area in the 25.06 ontology through their viral aetiology; they are cancers, and any "
    "pair built on them should not be read as an immunity-infection trade-off."
)

    diseaseId                                diseaseName  credible_sets  lead_variants
MONDO_0100096                                   COVID-19             49             16
MONDO_0024355      respiratory tract infectious disorder             12             10
MONDO_0004678                            dermatophytosis             11              6
  EFO_0007429                      peritonsillar abscess             10              7
MONDO_0001628                              tinea unguium              7              6
  EFO_0001054                                    leprosy              6              6
  EFO_0003030                                    abscess              6              5
MONDO_0002040                             dermatomycosis              6              6
  EFO_0003035                                 cellulitis              5              4
  EFO_0000183                          Hodgkins lymphoma              4              4
  EFO_0007512                              

# Step 3 — representatives

Chosen to cover both verdicts and both kinds of infection evidence, and to be interpretable: two
mycobacterial examples (leprosy) pointing opposite ways, the canonical TYK2 viral example, a barrier
gene, and a broad immune regulator. Every direction below is stated for the named effect allele.

In [8]:
REPRESENTATIVES = [
    ("TYK2", "19_10355447_C_T", "autoimmunity down, COVID-19 up on the same allele"),
    (
        "TYK2 P1104A",
        "19_10352442_G_C",
        "the classic protective allele: 38 immune credible sets, all negative, but its 3 COVID-19 "
        "credible sets report no beta — a coverage limit, not a concordant result",
    ),
    ("C1orf141 / IL23R", "1_67131436_G_A", "Crohn's disease down, leprosy up"),
    ("CCDC88B", "11_64340263_G_A", "autoimmune disease down, leprosy up"),
    ("LACC1", "13_43883789_A_G", "inflammatory disease up and leprosy up — a concordant counter-example"),
    ("FLG", "1_152313385_G_A", "filaggrin loss: atopic disease up and dermatophyte infection up"),
    ("SH2B3", "12_111446804_T_C", "broad immune regulator, 24 immune credible sets, all one way"),
    ("ABO", "9_133273813_C_T", "immune trait up, COVID-19 down"),
]

representative_rows = []
for label, variant_id, why in REPRESENTATIVES:
    rows = classified[(classified["variantId"] == variant_id) & ~classified["joint_study"]].copy()
    rows.insert(0, "label", label)
    representative_rows.append(rows)

    immune = rows[rows["class"] == "immune"]
    infection = rows[rows["class"] == "infection"]
    print("=" * 110)
    print(
        f"{label}  {variant_id}   effect allele {rows['effectAllele'].iloc[0]} "
        f"(other allele {rows['otherAllele'].iloc[0]})   — {why}"
    )
    print(
        f"  concordance over all {cs.loc[cs['variantId'] == variant_id, 'studyLocusId'].nunique()} "
        f"credible sets of this variant (paper formula): "
        f"{paper_concordance(cs[cs['variantId'] == variant_id]):.3f}"
    )
    print(
        f"  immune side: {immune['studyLocusId'].nunique()} credible sets, "
        f"{immune['diseaseId'].nunique()} diseases, sign {sign_of(immune['beta'])}"
    )
    print(
        f"  infection side: {infection['studyLocusId'].nunique()} credible sets, "
        f"{infection['diseaseId'].nunique()} diseases, sign {sign_of(infection['beta'])}"
    )
    print()
    print(
        rows[["class", "studyId", "trait", "diseaseId", "diseaseName", "originalBeta", "beta"]]
        .sort_values(["class", "beta"])
        .to_string(index=False)
    )
    print()

representatives = pd.concat(representative_rows, ignore_index=True)
representatives = representatives[
    [
        "label",
        "variantId",
        "effectAllele",
        "otherAllele",
        "class",
        "studyId",
        "trait",
        "diseaseId",
        "diseaseName",
        "originalBeta",
        "originalStandardError",
        "beta",
        "se",
        "nCases",
        "nControls",
        "studyLocusId",
    ]
]
representatives.to_csv(INTERMEDIATE + "immunity_infection_representatives-r1.csv", index=False)
print(f"written: {len(representatives)} association rows for {len(REPRESENTATIVES)} representatives")

TYK2  19_10355447_C_T   effect allele T (other allele C)   — autoimmunity down, COVID-19 up on the same allele
  concordance over all 13 credible sets of this variant (paper formula): 0.846
  immune side: 1 credible sets, 1 diseases, sign -
  infection side: 11 credible sets, 1 diseases, sign +

    class                   studyId                                                                trait     diseaseId         diseaseName  originalBeta      beta
   immune FINNGEN_R12_M13_PSORIARTH                                              Psoriatic arthropathies   EFO_0003778 psoriatic arthritis     -0.043374 -0.172382
infection              GCST90278684                                    COVID-19 with respiratory failure MONDO_0100096            COVID-19      0.103700  0.056691
infection              GCST90250835 COVID-19 critical illness or age-related macular degeneration (MTAG) MONDO_0100096            COVID-19      0.010000  0.063448
infection              GCST90134600                

# What this shows

**Genes.** 207 L2G-prioritised genes carry at least one immune-area and one infection-area credible
set; 167 of them survive dropping the trans-disease / MTAG studies that map a single credible set to
both areas. The largest are the expected immune regulators — IL12B (107 immune credible sets), STAT4
(102), TYK2 (82), IRF5 (81), TNFAIP3 (75) — but for almost all of them the infection side rests on
one or two credible sets.

**Same-allele comparisons are much rarer than gene-level overlap.** Only **59** gene × lead-variant
pairs (44 genes, 58 lead variants) have both an immune and an infection credible set **on the same
lead variant**, which is the only configuration where the two directions refer to the same effect
allele. Gene-level overlap is 207 genes; allele-exact overlap is 44. Dropping the trans-disease /
MTAG credible sets alone halves the count (122 → 59), because those studies put the identical beta on
both sides and manufacture concordance.

**Where the comparison is allele-exact, discordance is common.** 34 concordant, 16 discordant, 9
undetermined — **32% of determined pairs are discordant**, against 7.5% at the genome-wide variant
level. So balancing selection is not absent from the data; it is confined to the small set of loci
where both an immune trait and an infection trait have actually been measured on the same lead
variant.

**Representatives** (direction always for the named effect allele):

| gene | variant | effect allele | immune side | infection side | verdict |
|---|---|---|---|---|---|
| TYK2 | `19_10355447_C_T` | T | psoriatic arthritis −0.17 | COVID-19, 11 credible sets, +0.06 to +0.22 | discordant |
| TYK2 P1104A | `19_10352442_G_C` | C | 38 credible sets, all negative (psoriasis −0.27…−0.46, RA −0.18…−0.35, SLE −0.40, T1D −0.28/−0.31) | 3 COVID-19 credible sets, none reporting a beta; tonsillitis +0.19 | coverage limit, not concordance |
| C1orf141 / IL23R | `1_67131436_G_A` | A | Crohn's disease −0.27 | leprosy **+0.70** | discordant |
| CCDC88B | `11_64340263_G_A` | A | immune system disease −0.13, psoriasis −0.07, autoimmune disease −0.04/−0.05 | leprosy **+0.38** | discordant |
| LACC1 | `13_43883789_A_G` | G | Crohn's disease, 5 credible sets, +0.14 to +0.29 | leprosy **+1.11** | concordant |
| FLG | `1_152313385_G_A` | A | atopic eczema +0.35/+0.91, contact dermatitis +0.55, allergic disease +0.66 | dermatophytosis, dermatomycosis, tinea unguium +0.12 to +0.14 | concordant |
| SH2B3 | `12_111446804_T_C` | C | 19 credible sets, all negative (T1D −0.12…−0.29, celiac −0.16…−0.21, MS −0.10) | respiratory tract infection −0.04, prosthesis-related infection −0.03 | concordant |
| ABO | `9_133273813_C_T` | T | Graves disease +0.34 | COVID-19, 8 credible sets, −0.06 to −0.10 | discordant |

The two leprosy loci are the cleanest illustration of the point and of its limits: IL23R and CCDC88B
show the classic antagonistic pattern (autoimmune risk down, mycobacterial infection risk up), while
LACC1 — also a leprosy and Crohn's locus — is concordant on the same axis. Both patterns exist; the
corpus only lets us see them at the handful of loci where an infection GWAS was run.

**Caveats.** The infection side is dominated by COVID-19 (49 of the comparable credible sets, on 16
lead variants); several Hodgkin-lymphoma terms carry the infectious-disease area in the 25.06
ontology through their viral aetiology and are cancers, not infections; and the immune-system area is
broader than autoimmunity, so allergy and inflammatory terms (FLG's eczema, for example) are included
by design.